In [0]:
import uuid
from pyspark.sql.functions import col

In [0]:
SILVER_PATH = "/Volumes/workspace/legal_data/silver/legal_sections/"

silver_df = spark.read.format("delta").load(SILVER_PATH)

silver_df.count()

1281

In [0]:
def chunk_text(text, max_chars=1800, overlap=200):
    if not text:
        return []

    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]

        # try to end at sentence boundary
        if end < len(text):
            last_period = chunk.rfind(".")
            if last_period > 300:
                end = start + last_period + 1
                chunk = text[start:end]

        chunks.append(chunk.strip())
        start = end - overlap

    return chunks

In [0]:
from collections import Counter
import re

def extract_keywords(text, top_n=5):
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    common = Counter(words).most_common(top_n)
    return [w for w, _ in common]

In [0]:
records = []

for row in silver_df.collect():

    chunks = chunk_text(row.section_text)

    for i, chunk in enumerate(chunks):

        records.append({
            "chunk_id": str(uuid.uuid4()),
            "section_id": row.section_id,
            "doc_id": row.doc_id,
            "file_name": row.file_name,
            "category": row.category,
            "act_name": row.act_name,
            "section_number": row.section_number,
            "chunk_index": i,
            "chunk_text": chunk,
            "char_count": len(chunk),
            "keywords": extract_keywords(chunk),
            "created_at": row.created_at
        })

In [0]:
gold_df = spark.createDataFrame(records)

gold_df.display()

act_name,category,char_count,chunk_id,chunk_index,chunk_text,created_at,doc_id,file_name,keywords,section_id,section_number
Aadhaar Act 2016,acts,1707,69c42919-dda2-403b-8147-7a37258a291a,0,"( , ) ACT, 2016 ______ ______ 1. Short title, extent and commencement. 2. Definitions. 3. Aadhaar number. 3A. Aadhaar number of children. 4. Properties of Aadhaar number. 5. Special measures for issuance of Aadhaar number to certain category of persons. 6. Update of certain information. 7. Proof of Aadhaar number necessary for receipt of certain subsidies, benefits and services, etc. 8. Authentication of Aadhaar number. 8A. Offline verification of Aadhaar number. 9. Aadhaar number not evidence of citizenship or domicile, etc. 10. Central Identities Data Repository. 11. Establishment of Authority. 12. Composition of Authority. 13. Qualifications for appointment of Chairperson and Members of Authority. 14. Term of office and other conditions of service of Chairperson and Members. 15. Removal of Chairperson and Members. 16. Restrictions on Chairperson or Members on employment after cessation of office. 17. Functions of Chairperson. 1 18. Chief executive officer. 19. Meetings of Authority. 20. Vacancies, etc., not to invalidate proceedings of Authority. 21. Officers and other employees of Authority. 22. Transfer of assets, liabilities of Authority. 23. Powers and functions of Authority. 23A. Power of Authority to issue directions. , 24. Grants by Central Government. 25. Fund. 26. Accounts and audit. 27. Returns and annual report, etc. 28. Security and confidentiality of information. 29. Restriction on sharing information. 30. Biometric information deemed to be sensitive personal information. 31. Alteration of demographic information or biometric information. 32. Access to own information and records of requests for authentication. 33. Disclosure of information in certain cases. 33A.",2026-02-26T12:55:39.214Z,364cfa16-7717-4ebb-b565-3c4cfd49cd16,Aadhaar_Act_2016.pdf,"List(information, authority, aadhaar, number, chairperson)",0fa33fff-e182-4ca9-b0d3-6031e39dc928,null
Aadhaar Act 2016,acts,1796,1f8bc3b4-19e9-4475-b69b-c82a1da15bd5,1,"tion. 31. Alteration of demographic information or biometric information. 32. Access to own information and records of requests for authentication. 33. Disclosure of information in certain cases. 33A. Penalty for failure to comply with provisions of this Act, rules, regulations and directions. 33B. Power to adjudicate. 33C. Appeals to Appellate Tribunal. 33D. Procedure and powers of the Appellate Tribunal 33E. Appeal to Supreme Court of India. 33F. Civil court not to have jurisdiction. 2 34. Penalty for impersonation at time of enrolment. 35. Penalty for impersonation of Aadhaar number holder by changing demographic information or biometric information. 36. Penalty for impersonation. 37. Penalty for disclosing identity information. 38. Penalty for unauthorised access to the Central Identities Data Repository. 39. Penalty for tampering with data in Central Identities Data Repository. 40. Penalty for unauthorised use by requesting entity or offline verification-seeking entity. 41. Penalty for non-compliance with intimation requirements. 42. General penalty. 43. Offences by companies. 44. Act to apply for offence or contravention committed outside India. 45. Power to investigate offences. 46. Penalties not to interfere with other punishments. 47. Cognizance of offences. 48. Power of Central Government to supersede Authority. 49. Members, officers, etc., to be public servants. 50. Power of Central Government to issue directions. 50A. Exemption from tax on income. 51. Delegation. 52. Protection of action taken in good faith. 53. Power of Central Government to make rules. 54. Power of Authority to make regulations. 55. Laying of rules and regulations before Parliament. 56. Application of other laws not barred. 57. [Omitted.]. 58. Power to remove difficulties. 59. Savings.",2026-02-26T12:55:39.214Z,364cfa16-7717-4ebb-b565-

In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .save(GOLD_PATH)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_legal_chunks")


##### Use this after this :
gold_df.write \
    .mode("append") \
    .saveAsTable("workspace.default.gold_legal_chunks")

In [0]:
%sql
CREATE TABLE workspace.default.gold_legal_chunks (
  act_name STRING,
  category STRING,
  char_count BIGINT,
  chunk_id STRING,
  chunk_index BIGINT,
  chunk_text STRING,
  created_at TIMESTAMP,
  doc_id STRING,
  file_name STRING,
  keywords ARRAY<STRING>,
  section_id STRING,
  section_number STRING
)
USING DELTA;

In [0]:
gold_df.write.mode("append").saveAsTable("workspace.default.gold_legal_chunks")

In [0]:
%sql
select count(*) from workspace.default.gold_legal_chunks

count(*)
5194


In [0]:
gold_df.count()

5194

In [0]:
gold_df.select("char_count").describe().display()

summary,char_count
count,5194
mean,1503.6432422025414
stddev,385.9097189424425
min,1
max,1800


In [0]:
gold_df.select("chunk_text").limit(10).display()

chunk_text
"( , ) ACT, 2016 ______ ______ 1. Short title, extent and commencement. 2. Definitions. 3. Aadhaar number. 3A. Aadhaar number of children. 4. Properties of Aadhaar number. 5. Special measures for issuance of Aadhaar number to certain category of persons. 6. Update of certain information. 7. Proof of Aadhaar number necessary for receipt of certain subsidies, benefits and services, etc. 8. Authentication of Aadhaar number. 8A. Offline verification of Aadhaar number. 9. Aadhaar number not evidence of citizenship or domicile, etc. 10. Central Identities Data Repository. 11. Establishment of Authority. 12. Composition of Authority. 13. Qualifications for appointment of Chairperson and Members of Authority. 14. Term of office and other conditions of service of Chairperson and Members. 15. Removal of Chairperson and Members. 16. Restrictions on Chairperson or Members on employment after cessation of office. 17. Functions of Chairperson. 1 18. Chief executive officer. 19. Meetings of Authority. 20. Vacancies, etc., not to invalidate proceedings of Authority. 21. Officers and other employees of Authority. 22. Transfer of assets, liabilities of Authority. 23. Powers and functions of Authority. 23A. Power of Authority to issue directions. , 24. Grants by Central Government. 25. Fund. 26. Accounts and audit. 27. Returns and annual report, etc. 28. Security and confidentiality of information. 29. Restriction on sharing information. 30. Biometric information deemed to be sensitive personal information. 31. Alteration of demographic information or biometric information. 32. Access to own information and records of requests for authentication. 33. Disclosure of information in certain cases. 33A."
"tion. 31. Alteration of demographic information or biometric information. 32. Access to own information and records of requests for authentication. 33. Disclosure of information in certain cases. 33A. Penalty for failure to comply with provisions of this Act, rules, regulations and directions. 33B. Power to adjudicate. 33C. Appeals to Appellate Tribunal. 33D. Procedure and powers of the Appellate Tribunal 33E. Appeal to Supreme Court of India. 33F. Civil court not to have jurisdiction. 2 34. Penalty for impersonation at time of enrolment. 35. Penalty for impersonation of Aadhaar number holder by changing demographic information or biometric information. 36. Penalty for impersonation. 37. Penalty for disclosing identity information. 38. Penalty for unauthorised access to the Central Identities Data Repository. 39. Penalty for tampering with data in Central Identities Data Repository. 40. Penalty for unauthorised use by requesting entity or offline verification-seeking entity. 41. Penalty for non-compliance with intimation requirements. 42. General penalty. 43. Offences by companies. 44. Act to apply for offence or contravention committed outside India. 45. Power to investigate offences. 46. Penalties not to interfere with other punishments. 47. Cognizance of offences. 48. Power of Central Government to supersede Authority. 49. Members, officers, etc., to be public servants. 50. Power of Central Government to issue directions. 50A. Exemption from tax on income. 51. Delegation. 52. Protection of action taken in good faith. 53. Power of Central Government to make rules. 54. Power of Authority to make regulations. 55. Laying of rules and regulations before Parliament. 56. Application of other laws not barred. 57. [Omitted.]. 58. Power to remove difficulties. 59. Savings."
"Power of Authority to make regulations. 55. Laying of rules and regulations before Parliament. 56. Application of other laws not barred. 57. [Omitted.]. 58. Power to remove difficulties. 59. Savings. 3 ( , ) ACT, 2016 . 18 OF 2016 [25th March, 2016.] An Act to provide for, as a good governance, efficient, transparent, and targeted delivery of subsidies, benefits and services, the expenditure for which is incurred from the Consolidated Fund of India, 1[or the Consolidated Fun

In [0]:
gold_df.select("section_number").tail(10)

[Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VII'),
 Row(section_number='Chapter VI'),
 Row(section_number='Chapter VI'),
 Row(section_number='Chapter II')]